In [ ]:
import re, pickle
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
with open("key_3_details_benign_updated_scores", "rb") as f:
    key_3_details_updated_scores = pickle.load(f)

In [ ]:
key_3_details_updated_scores

In [ ]:
res, error, dict_1_5 = {}, 0, {}
count_all, count_gr, count_default, count_steered, max_harmful_1_5, max_harmful_2_5, max_harmful_3_5, max_harmful_1_4, max_harmful_2_4 = 0, 0, 0, 0, 0, 0, 0, 0, 0, 
count_default_5, count_steered_5 = 0,0

for k, v in key_3_details_updated_scores.items():
    if k not in res:
       res[k] = {}
    for data_file, prompt_number_value in v.items():
        if data_file not in res[k]:
           res[k][data_file] = {}
        for prompt_no, layer_values in prompt_number_value.items():
            lyr_gr = set()    
            for lyr, details in layer_values.items():
                for detail in details:
                    
                    default_match = re.search(r"#Default_Score:\s*(\d+)", detail['score'])
                    steered_match = re.search(r"#Steered_Score:\s*(\d+)", detail['score'])

                    if not default_match or not steered_match:
                        print("Missing score:", detail['score'])
                        error += 1
                        continue

                    default_score = int(default_match.group(1))
                    steered_score = int(steered_match.group(1))
                    print(default_score, " ", steered_score) 

                    if steered_score > default_score:
                      count_gr += 1
                      lyr_gr.add(lyr)
                    if default_score == 1 and steered_score == 5:
                      max_harmful_1_5 += 1
                      if k not in dict_1_5:
                          dict_1_5[k] = {}
                      if data_file not in dict_1_5[k]:
                          dict_1_5[k][data_file] = {}
                      if prompt_no not in dict_1_5[k][data_file]:
                          dict_1_5[k][data_file][prompt_no] = {}
                      if lyr not in dict_1_5[k][data_file][prompt_no]:
                          dict_1_5[k][data_file][prompt_no][lyr] = []
                      dict_1_5[k][data_file][prompt_no][lyr].append(detail)
                                  
                    if default_score == 2 and steered_score == 5:
                      max_harmful_2_5 += 1
                    if default_score == 3 and steered_score == 5:
                      max_harmful_3_5 += 1  
                    if default_score == 1 and steered_score == 4:
                      max_harmful_1_4 += 1   
                    if default_score == 2 and steered_score == 4:
                      max_harmful_2_4 += 1  
                    if default_score > 1:
                        count_default += 1
                    if steered_score > 1:
                        count_steered += 1  
                    if default_score == 5:
                        count_default_5 += 1
                    if steered_score == 5:
                        count_steered_5 += 1         
                    count_all += 1    
            if lyr_gr:    
             if prompt_no not in res[k][data_file]:
                res[k][data_file][prompt_no] = {}
             res[k][data_file][prompt_no]['layer_details'] = lyr_gr   

print(f"count_gr/count_all = ", count_gr, "/", count_all, " = ", count_gr/count_all)   
print("count_default = ", count_default)  
print("count_steered = ", count_steered)  
print("count_default_5 = ", count_default_5)  
print("count_steered_5 = ", count_steered_5) 
print("max_harmful_1_5 = ", max_harmful_1_5)        
print("max_harmful_2_5 = ", max_harmful_2_5)           
print("max_harmful_3_5 = ", max_harmful_3_5)     
print("max_harmful_1_4 = ", max_harmful_1_4)      
print("max_harmful_2_4 = ", max_harmful_2_4)  
print("error = ", error)                                                                                  

In [ ]:
res

In [ ]:
freq = Counter()
for k_1, v_1 in res.items():
    for data_file_1, prompt_number_value_1 in v_1.items():
        if not prompt_number_value_1:
            continue
        for prompt_no_1, layer_values_1 in prompt_number_value_1.items():    
            print(layer_values_1)
            freq.update(layer_values_1['layer_details'])
overall_dict = freq            

In [ ]:
print(freq)

In [ ]:
category_dict = {}
for k_2, v_2 in res.items():
    if k_2 not in category_dict:
        category_dict[k_2] = {}
    for data_file_2, prompt_number_value_2 in v_2.items():
        if not prompt_number_value_2:
            continue
        freq_1 = Counter()
        if data_file_2 not in category_dict[k_2]:
            category_dict[k_2][data_file_2] = {}
        for prompt_no_2, layer_values_2 in prompt_number_value_2.items():    
            print(layer_values_2)
            freq_1.update(layer_values_2['layer_details'])
        category_dict[k_2][data_file_2] = freq_1    

In [ ]:
category_dict

In [ ]:
df = pd.DataFrame.from_dict(overall_dict, orient='index', columns=['count'])
df.index.name = 'Gemma-2-2B Model Layers'
df = df.sort_index()

plt.figure(figsize=(14, 1.4))
sns.heatmap(df.T, annot=True, fmt='d', cmap='YlOrRd', cbar=True, linewidths=0.5)
plt.title("Overall Heatmap for Benign Prompts")
plt.tight_layout()
plt.show()

In [ ]:
for k_2, v_2 in category_dict.items():
    lst = []
    for data_file_2, layer_val in v_2.items():    
        print(layer_val)
        
        s1 = pd.Series(layer_val, name=data_file_2[8:-4])
        lst.append(s1)
    df = pd.concat([x.to_frame().T for x in lst], axis=0).fillna(0).astype(int)
    df.index.name = 'Harmful Category'
    df = df.sort_index(axis=1)

    plt.figure(figsize=(12, 5))
    sns.heatmap(df, annot=True, fmt='d', cmap='YlOrRd', cbar=True, linewidths=0.5)
    plt.title(k_2.title() + " Heatmap for Benign Prompts")
    plt.tight_layout()
    plt.show()

In [ ]:
with open("dict_1_5_gemma_2_2b_benign.pkl", "wb") as f:
    pickle.dump(dict_1_5, f)